In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("used_cars_cleaned_08.csv")

# Explicitly select information available before estimating the price.
feature_cols = [
    "make", "model", "version", "mileage_miles", "reg_year",
    "fuel", "transmission", "power_hp", "body_type", "body_color",
    "doors", "seats", "previous_owners", "drivetrain", "emission_class",
    "annual_tax_gbp", "seller_type", "seller_city", "seller_county",
    "co2_gmile",
]

X = df.loc[:, feature_cols].copy()
y = df["price_gbp"].copy()

# Keep identical predictor rows together, even if their prices differ.
groups = pd.util.hash_pandas_object(X, index=False)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(X, y, groups=groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
train_groups = groups.iloc[train_idx]

assert not groups.iloc[val_idx].isin(train_groups).any()

# Precompute folds for the later searches. These positions refer to X_train.
# X_train_text also works with these folds if it preserves the same row order.
grouped_cv = list(
    GroupKFold(n_splits=3).split(X_train, y_train, groups=train_groups)
)
cv = grouped_cv

numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols),
])

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

linear_model.fit(X_train, y_train)

train_predictions = linear_model.predict(X_train)
val_predictions = linear_model.predict(X_val)

results = []
for split, actual, predicted in [
    ("Training", y_train, train_predictions),
    ("Validation", y_val, val_predictions),
]:
    results.append({
        "Dataset": split,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
    })

print(f"Training listings: {len(X_train):,}")
print(f"Validation listings: {len(X_val):,}")
display(pd.DataFrame(results).round(3))

comparison = pd.DataFrame({
    "Actual Price": y_val.to_numpy(),
    "Predicted Price": val_predictions,
}, index=y_val.index)
comparison["Absolute Error"] = (
    comparison["Actual Price"] - comparison["Predicted Price"]
).abs()

display(comparison.head(10).round(2))
print("Negative validation predictions:", (val_predictions < 0).sum())

Training listings: 289,266
Validation listings: 72,324


,Dataset,MAE (£),RMSE (£),R²
0,Training,1605.726,7824.662,0.945
1,Validation,2180.196,13021.904,0.791


,Actual Price,Predicted Price,Absolute Error
6,26499,27689.67,1190.67
17,1499,-3319.76,4818.76
21,2795,4724.49,1929.49
27,3795,3627.68,167.32
28,7800,9112.03,1312.03
29,16895,17334.75,439.75
32,5490,7418.85,1928.85
33,41388,41070.95,317.05
38,3995,5266.42,1271.42
41,13168,12908.88,259.12


Negative validation predictions: 1728


In [2]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(solver="lsqr"))
])

param_grid = {
    "model__alpha": [0.1, 1, 10, 100, 1000]
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    verbose=1,
    refit=True
)

# Tune using training data only
search.fit(X_train, y_train)

best_ridge = search.best_estimator_

print("Best parameters:", search.best_params_)
print("Best cross-validation MAE (£):", round(-search.best_score_, 2))

# Evaluate the selected model on validation data
predictions = best_ridge.predict(X_val)

print("\nValidation results")
print("MAE (£):", round(mean_absolute_error(y_val, predictions), 2))
print("RMSE (£):", round(np.sqrt(
    mean_squared_error(y_val, predictions)
), 2))
print("R²:", round(r2_score(y_val, predictions), 4))

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best parameters: {'model__alpha': 0.1}
Best cross-validation MAE (£): 2312.37

Validation results
MAE (£): 2235.42
RMSE (£): 13071.57
R²: 0.7889


In [3]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Exclude missing-value indicators from polynomial expansion
poly_cols = [
    col for col in numeric_cols
    if not col.endswith("_missing")
]

indicator_cols = [
    col for col in numeric_cols
    if col.endswith("_missing")
]

polynomial_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler())
])

poly_preprocessor = ColumnTransformer([
    ("numeric_poly", polynomial_pipeline, poly_cols),
    ("indicators", "passthrough", indicator_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

poly_model = Pipeline([
    ("preprocessor", poly_preprocessor),
    ("model", Ridge(alpha=100, solver="lsqr"))
])

# Use the same split as previous models for a fair comparison
poly_model.fit(X_train, y_train)

results = []

for name, X_part, y_part in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val)
]:
    predictions = poly_model.predict(X_part)

    results.append({
        "Dataset": name,
        "MAE (£)": mean_absolute_error(y_part, predictions),
        "RMSE (£)": np.sqrt(mean_squared_error(y_part, predictions)),
        "R²": r2_score(y_part, predictions)
    })

display(pd.DataFrame(results).round(3))

,Dataset,MAE (£),RMSE (£),R²
0,Training,3455.490,22311.723,0.554
1,Validation,3448.116,15361.762,0.708


In [4]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

param_grid = {
    "preprocessor__numeric_poly__polynomial__degree": [1, 2],
    "model__alpha": [0.1, 1, 10, 100, 1000]
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)

poly_search = GridSearchCV(
    estimator=poly_model,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1,
    refit=True,
    error_score="raise"
)

# Tune on training data only
poly_search.fit(X_train, y_train)

best_poly_model = poly_search.best_estimator_

print("Best parameters:", poly_search.best_params_)
print("Best CV MAE (£):", round(-poly_search.best_score_, 2))

# Compare training and validation performance
results = []

for name, X_part, y_part in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val)
]:
    predictions = best_poly_model.predict(X_part)

    results.append({
        "Dataset": name,
        "MAE (£)": mean_absolute_error(y_part, predictions),
        "RMSE (£)": np.sqrt(mean_squared_error(y_part, predictions)),
        "R²": r2_score(y_part, predictions)
    })

display(pd.DataFrame(results).round(3))

# Show all parameter combinations, best first
tuning_results = pd.DataFrame(poly_search.cv_results_)
tuning_results["CV MAE (£)"] = -tuning_results["mean_test_score"]

display(
    tuning_results[[
        "param_preprocessor__numeric_poly__polynomial__degree",
        "param_model__alpha",
        "CV MAE (£)",
        "rank_test_score"
    ]]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters: {'model__alpha': 1, 'preprocessor__numeric_poly__polynomial__degree': 2}
Best CV MAE (£): 2253.29


,Dataset,MAE (£),RMSE (£),R²
0,Training,2131.245,9827.773,0.914
1,Validation,2375.482,12461.338,0.808


,param_preprocessor__numeric_poly__polynomial__degree,param_model__alpha,CV MAE (£),rank_test_score
0,2,1.0,2253.292730,1
1,1,0.1,2310.697930,2
2,1,1.0,2462.604668,3
3,2,10.0,2857.167125,4
4,2,0.1,3077.160683,5
5,1,10.0,3288.404575,6
6,2,100.0,3687.594636,7
7,2,1000.0,4489.838420,8
8,1,100.0,4609.627768,9
9,1,1000.0,5984.967891,10


In [5]:
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Use the polynomial degree selected by your previous search
sgd_model = Pipeline([
    ("preprocessor", best_poly_model.named_steps["preprocessor"]),
    ("model", SGDRegressor(
        loss="squared_error",
        penalty="l2",
        max_iter=3000,
        tol=1e-4,
        average=True,
        random_state=42
    ))
])

param_grid = {
    "model__alpha": [0.0001, 0.001, 0.01],
    "model__learning_rate": ["adaptive"],
    "model__eta0": [0.0001, 0.001]
}

sgd_search = GridSearchCV(
    sgd_model,
    param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1
)

sgd_search.fit(X_train, y_train)

predictions = sgd_search.predict(X_val)

print("Best parameters:", sgd_search.best_params_)
print("Validation MAE (£):",
      round(mean_absolute_error(y_val, predictions), 2))
print("Validation RMSE (£):",
      round(np.sqrt(mean_squared_error(y_val, predictions)), 2))
print("Validation R²:", round(r2_score(y_val, predictions), 4))

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best parameters: {'model__alpha': 0.0001, 'model__eta0': 0.001, 'model__learning_rate': 'adaptive'}
Validation MAE (£): 3114.28
Validation RMSE (£): 14794.66
Validation R²: 0.7296


In [6]:
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

log_ridge = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", Ridge(solver="lsqr"))
    ]),
    func=np.log1p,
    inverse_func=np.expm1
)

log_search = GridSearchCV(
    log_ridge,
    param_grid={
        "regressor__model__alpha": [0.01, 0.1, 1, 10, 100]
    },
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1
)

log_search.fit(X_train, y_train)

# Predictions are already in pounds
predictions = log_search.predict(X_val)

print("Best parameters:", log_search.best_params_)
print("Validation MAE (£):",
      round(mean_absolute_error(y_val, predictions), 2))
print("Validation RMSE (£):",
      round(np.sqrt(mean_squared_error(y_val, predictions)), 2))
print("Validation R²:", round(r2_score(y_val, predictions), 4))

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best parameters: {'regressor__model__alpha': 0.1}
Validation MAE (£): 1610.46
Validation RMSE (£): 12257.21
Validation R²: 0.8144


In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Work on copies of the existing split
X_train_text = X_train.copy()
X_val_text = X_val.copy()

for frame in [X_train_text, X_val_text]:
    frame["version"] = frame["version"].fillna("").astype(str)

numeric_cols = X_train_text.select_dtypes(
    include="number"
).columns.tolist()

# Exclude version: it will be handled by CountVectorizer
other_categorical_cols = [
    col for col in X_train_text.select_dtypes(
        include=["object", "string", "category"]
    ).columns
    if col != "version"
]

text_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),

    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), other_categorical_cols),

    # A string selector passes a 1D text column to CountVectorizer
    ("version_text", CountVectorizer(
        lowercase=True,
        strip_accents="unicode",
        token_pattern=r"(?u)\b\w[\w.-]*\b",
        min_df=3,
        max_features=15000
    ), "version")
])

text_model = Pipeline([
    ("preprocessor", text_preprocessor),
    ("model", Ridge(solver="lsqr"))
])

text_search = GridSearchCV(
    text_model,
    param_grid={
        "preprocessor__version_text__ngram_range": [(1, 1), (1, 2)],
        "model__alpha": [0.01, 0.1, 1, 10]
    },
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1
)

text_search.fit(X_train_text, y_train)

predictions = text_search.predict(X_val_text)

print("Best parameters:", text_search.best_params_)
print("Validation MAE (£):",
      round(mean_absolute_error(y_val, predictions), 2))
print("Validation RMSE (£):",
      round(np.sqrt(mean_squared_error(y_val, predictions)), 2))
print("Validation R²:", round(r2_score(y_val, predictions), 4))

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best parameters: {'model__alpha': 1, 'preprocessor__version_text__ngram_range': (1, 2)}
Validation MAE (£): 2491.25
Validation RMSE (£): 13184.33
Validation R²: 0.7853


In [8]:
err = (y_val - best_ridge.predict(X_val)).abs().sort_values(ascending=False)
X_val.loc[err.index[:20]].assign(actual=y_val, err=err)

,make,model,version,mileage_miles,reg_year,fuel,transmission,power_hp,body_type,body_color,...,previous_owners,drivetrain,emission_class,annual_tax_gbp,seller_type,seller_city,seller_county,co2_gmile,actual,err
270550,Aston Martin,Valour,5.2 V12 BiTurbo Euro 6 (s/s) 2dr,1100,2024,Petrol,Manual,715,Coupe,Unknown,...,1,Rear Wheel Drive,Euro 6,200.0,Trade,Pangbourne,West Berkshire,547,1785000,1.614212e+06
20590,Porsche,911,993 GT2,21727,1995,Petrol,Manual,444,Coupe,Silver,...,3,Rear Wheel Drive,Euro 1,200.0,Trade,Westminster,Middlesex,497,1600000,1.493052e+06
311536,Aston Martin,Valour,5.2 V12 BiTurbo Euro 6 (s/s) 2dr,250,2025,Petrol,Manual,715,Coupe,Black,...,1,Rear Wheel Drive,Euro 6,200.0,Trade,Preston,Lancashire,547,1499995,1.338091e+06
113656,Rimac,Nevera,120kWh GT Auto 4WD 2dr,735,2023,Electric,Automatic,1888,Coupe,Green,...,1,Four Wheel Drive,Euro 6,200.0,Trade,London,London,0,1500000,1.313897e+06
36377,Aston Martin,Valhalla,Unknown,193,2026,Petrol Plug-in Hybrid,Automatic,265,Unknown,Yellow,...,1,Unknown,Euro 6,200.0,Trade,Newtownabbey,County Antrim,63,1099995,8.887975e+05
280161,Rolls-Royce,Phantom,6.7 V12 Drophead Coupe Auto Euro 6 2dr,479,2017,Petrol,Automatic,460,Convertible,Red,...,2,Rear Wheel Drive,Euro 6,200.0,Trade,London,Islington,558,849000,5.514217e+05
208083,Ferrari,599,6.0 V12 GTB Fiorano Coupe 2dr Petrol F1 Euro 4...,22700,2009,Petrol,Automatic,612,Coupe,Red,...,2,Rear Wheel Drive,Euro 4,790.0,Private,Finsbury Park,Haringey,789,159000,3.588409e+05
166650,Ferrari,488 Pista,3.9T V8 Spider F1 DCT Euro 6 (s/s) 2dr,1000,2019,Petrol,Automatic,710,Convertible,Red,...,2,Rear Wheel Drive,Euro 6,200.0,Trade,Huddersfield,West Yorkshire,465,999995,3.378904e+05
343864,Rolls-Royce,Ghost,6.6 V12 Auto Euro 6 4dr,22000,2015,Petrol,Automatic,563,Saloon,Blue,...,4,Rear Wheel Drive,Euro 6,790.0,Private,Barnsley,South Yorkshire,526,420000,3.187146e+05
74588,Rolls-Royce,Wraith,6.6 V12 Black Badge Auto Euro 6 2dr,1200,2023,Petrol,Automatic,591,Coupe,Black,...,1,Rear Wheel Drive,Euro 6,200.0,Trade,Wilmslow,Cheshire,578,495000,2.984453e+05


In [9]:
from sklearn.base import clone
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

svd_linear = Pipeline([
    ("preprocessor", clone(text_preprocessor)),
    ("reduction", TruncatedSVD(random_state=42)),
    ("model", LinearRegression())
])

# Dimensionality reduction can be expensive: start small
svd_search = GridSearchCV(
    svd_linear,
    param_grid={
        "reduction__n_components": [50, 100, 200]
    },
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    verbose=1,
    error_score="raise"
)

svd_search.fit(X_train_text, y_train)

best_svd_model = svd_search.best_estimator_
predictions = best_svd_model.predict(X_val_text)

print("Best parameters:", svd_search.best_params_)
print("Validation MAE (£):",
      round(mean_absolute_error(y_val, predictions), 2))
print("Validation RMSE (£):",
      round(np.sqrt(mean_squared_error(y_val, predictions)), 2))
print("Validation R²:", round(r2_score(y_val, predictions), 4))

variance = (
    best_svd_model.named_steps["reduction"]
    .explained_variance_ratio_.sum()
)
print("Feature variance retained:", f"{variance:.1%}")

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best parameters: {'reduction__n_components': 100}
Validation MAE (£): 6707.07
Validation RMSE (£): 20602.26
Validation R²: 0.4757
Feature variance retained: 80.5%


In [10]:
from pathlib import Path

import joblib
from sklearn.utils.validation import check_is_fitted

check_is_fitted(linear_model)

model_path = Path("linear_reg.joblib")
joblib.dump(linear_model, model_path, compress=3)

print(f"Saved: {model_path.resolve()}")

Saved: D:\Msc DS\Machine Learning\Project\Sample Project\linear_reg.joblib
